# ARIMA Document Ingestion Forecast - Part 3
## Model Training and Forecasting

**Objective**: Build, train, and evaluate the ARIMA model, then generate forecasts.

**What we'll do in this notebook**:
1. Load data and parameters from previous notebooks
2. Train the ARIMA(5,1,2) model on training data
3. Evaluate model performance on test set
4. Retrain on full dataset for production forecasting
5. Generate next-day forecast with confidence intervals

**Prerequisites**: 
- Complete Notebook 1 (Data Preparation and EDA)
- Complete Notebook 2 (ACF/PACF Parameter Selection)

**Output**: Production-ready forecast with uncertainty estimates

In [0]:
# ============================================================
# LOAD PREPARED DATA AND PARAMETERS
# ============================================================
# Load everything prepared in the previous notebooks

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
from datetime import timedelta
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Load ARIMA parameters determined in Notebook 2
params_path = '/Workspace/Users/areebatanveerselling@gmail.com/ArimaForecasting/arima_parameters.pkl'
with open(params_path, 'rb') as f:
    params = pickle.load(f)

# Load prepared time-indexed dataframe
data_path = '/Workspace/Users/areebatanveerselling@gmail.com/ArimaForecasting/prepared_data.pkl'
with open(data_path, 'rb') as f:
    df_ts = pickle.load(f)

print("✓ Data and parameters loaded successfully")
print(f"\nARIMA Parameters:")
print(f"  p (AR order): {params['p']}")
print(f"  d (Differencing): {params['d']}")
print(f"  q (MA order): {params['q']}")
print(f"\nDataset: {len(df_ts)} days from {df_ts.index.min()} to {df_ts.index.max()}")

# Split into train/test using saved parameters
train_size = params['train_size']
train_data = df_ts[:train_size]
test_data = df_ts[train_size:]

print(f"\nTrain: {len(train_data)} days, Test: {len(test_data)} days")

In [0]:
# ============================================================
# ARIMA MODEL SPECIFICATION: ARIMA(p, d, q)
# ============================================================

# Parameter Recap:
# p = 5: AutoRegressive order
#   - Model uses the past 5 days to predict today
#   - Each past day gets a coefficient (weight) showing its importance
#   - Based on PACF plot showing significant correlations up to lag 5

# d = 1: Differencing order (Integration)
#   - Takes first difference: today's value minus yesterday's value
#   - Makes the series "stationary" (constant mean and variance over time)
#   - Removes trend so model can focus on patterns
#   - Formula: diff = value(t) - value(t-1)

# q = 2: Moving Average order
#   - Uses the past 2 forecast errors to improve predictions
#   - "Error" = actual value - predicted value from previous step
#   - Helps smooth out random fluctuations and noise
#   - Based on ACF plot showing decay pattern

# ============================================================
# MODEL TRAINING PROCESS
# ============================================================
# 1. Apply differencing (d=1) to make data stationary
# 2. Fit AR coefficients (p=5) using Maximum Likelihood Estimation
# 3. Fit MA coefficients (q=2) to model forecast errors
# 4. Optimize all parameters to minimize prediction error

print(f"Training ARIMA({params['p']},{params['d']},{params['q']}) model...")
print("This may take 30-60 seconds as the model optimizes parameters...\n")

model = ARIMA(train_data['number_documents'], order=(params['p'], params['d'], params['q']))
model_fit = model.fit()

print("\n" + "="*60)
print("MODEL SUMMARY")
print("="*60)
print(model_fit.summary())
print("\n" + "="*60)
print("KEY METRICS TO CHECK:")
print("="*60)
print("  - AIC (Akaike Information Criterion): Lower is better")
print("    Balances model fit vs complexity. Use to compare models.")
print("\n  - Coefficients (ar.L1, ar.L2... ma.L1, ma.L2):")
print("    Show the weight/importance of each lag and error term")
print("\n  - P>|z| (p-values): Should be < 0.05 for significant coefficients")
print("    Values > 0.05 suggest that parameter may not be needed")
print("\n  - Ljung-Box Q: Tests if residuals are random (want Prob(Q) > 0.05)")
print("    If Prob(Q) < 0.05, model may be missing patterns")

In [0]:
# ============================================================
# MODEL EVALUATION: Testing on Unseen Data
# ============================================================
# The model was trained on 970 days. Now we test it on 30 days
# it has NEVER seen to evaluate real-world performance.

# Generate predictions for test period (30 days)
# forecast() produces point estimates without updating the model
test_predictions = model_fit.forecast(steps=len(test_data))

# ============================================================
# EVALUATION METRICS
# ============================================================

# 1. MAE (Mean Absolute Error)
#    - Average absolute difference between actual and predicted
#    - Same units as original data (number of documents)
#    - Easy to interpret: "On average, we're off by X documents"
#    - Not sensitive to outliers
mae = mean_absolute_error(test_data['number_documents'], test_predictions)

# 2. RMSE (Root Mean Squared Error)
#    - Square root of average squared differences
#    - Penalizes large errors more heavily than MAE
#    - Also in same units as original data
#    - Higher than MAE when large errors exist
rmse = np.sqrt(mean_squared_error(test_data['number_documents'], test_predictions))

# 3. MAPE (Mean Absolute Percentage Error)
#    - Average percentage error (scale-independent)
#    - Good for comparing across different datasets/scales
#    - Expressed as %: easier for non-technical stakeholders
#    - <10% = excellent, 10-20% = good, >20% = needs improvement
mape = np.mean(np.abs((test_data['number_documents'] - test_predictions) / test_data['number_documents'])) * 100

print("="*60)
print("MODEL PERFORMANCE ON TEST SET (30 DAYS)")
print("="*60)
print(f"  Mean Absolute Error (MAE): {mae:.2f} documents")
print(f"    → On average, predictions are off by {mae:.0f} documents")
print(f"\n  Root Mean Squared Error (RMSE): {rmse:.2f} documents")
print(f"    → Penalizes large errors more. RMSE > MAE means some big misses exist")
print(f"\n  Mean Absolute Percentage Error (MAPE): {mape:.2f}%")
if mape < 10:
    print(f"    → Excellent accuracy! Predictions within {mape:.1f}% of actual values")
elif mape < 20:
    print(f"    → Good accuracy. Predictions within {mape:.1f}% of actual values")
else:
    print(f"    → Model needs improvement. Consider adjusting parameters.")
print("="*60)

In [0]:
# ============================================================
# VISUALIZATION: Actual vs Predicted
# ============================================================
# Visual inspection is crucial - metrics alone don't show everything
# Look for:
#   - Systematic bias (always over/under predicting)
#   - Pattern mismatches (missing peaks/valleys)
#   - Lag (predictions delayed compared to actual)

plt.figure(figsize=(14, 6))
plt.plot(test_data.index, test_data['number_documents'], 
         label='Actual', marker='o', linewidth=2, markersize=6, color='#2E86AB')
plt.plot(test_data.index, test_predictions, 
         label='Predicted', marker='s', linewidth=2, markersize=5, 
         alpha=0.7, color='#A23B72', linestyle='--')
plt.title('ARIMA Model: Actual vs Predicted (Test Set)', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Number of Documents', fontsize=12)
plt.legend(fontsize=11, loc='upper left')
plt.grid(True, alpha=0.3, linestyle=':')
plt.tight_layout()
plt.show()

print("\nVisual Check: Do predicted values follow the actual trend?")
print("  - Close alignment = good model fit")
print("  - Consistent gap = systematic bias")
print("  - Predictions lag behind actuals = model too slow to react")

In [0]:
# ============================================================
# FINAL FORECAST: Retraining on Full Dataset
# ============================================================
# Why retrain on full data?
#   1. We validated the model on test set - it performs well
#   2. For production forecasts, use ALL available data
#   3. More data = better parameter estimates = better predictions
#   4. The 30 test days contain useful patterns we should learn from

print("Retraining ARIMA(5,1,2) model on full dataset (1000 days)...")
final_model = ARIMA(df_ts['number_documents'], order=(params['p'], params['d'], params['q']))
final_model_fit = final_model.fit()

print("✓ Model retrained successfully on full dataset")
print(f"\nTraining data: {len(df_ts)} days")
print(f"Date range: {df_ts.index.min()} to {df_ts.index.max()}")
print(f"\nModel is now ready for production forecasting")

In [0]:
# ============================================================
# POINT FORECAST: Single Best Estimate
# ============================================================
# forecast(steps=1) predicts 1 day ahead
# This is a "point estimate" - the most likely value
# But reality is uncertain! That's why we also need confidence intervals.

next_day_forecast = final_model_fit.forecast(steps=1)[0]
next_date = df_ts.index[-1] + timedelta(days=1)

print(f"\n{'='*60}")
print(f"NEXT-DAY FORECAST")
print(f"{'='*60}")
print(f"Date: {next_date.strftime('%Y-%m-%d')} ({next_date.strftime('%A')})")
print(f"Predicted Number of Documents: {int(next_day_forecast)}")
print(f"{'='*60}")

In [0]:
# ============================================================
# CONFIDENCE INTERVALS: Quantifying Uncertainty
# ============================================================
# Forecasts are never 100% certain. Confidence intervals show the range
# of plausible values given the model's uncertainty.
#
# 95% Confidence Interval means:
#   - "We are 95% confident the true value falls in this range"
#   - If we made 100 forecasts, ~95 would have actuals within the interval
#   - Wider interval = more uncertainty
#   - Interval width grows as we forecast further into the future

forecast_result = final_model_fit.get_forecast(steps=1)
forecast_ci = forecast_result.conf_int(alpha=0.05)  # alpha=0.05 for 95% CI

ci_lower = int(forecast_ci.iloc[0, 0])
ci_upper = int(forecast_ci.iloc[0, 1])
ci_width = ci_upper - ci_lower

print(f"\n95% Confidence Interval:")
print(f"  Lower bound: {ci_lower} documents")
print(f"  Upper bound: {ci_upper} documents")
print(f"  Interval width: {ci_width} documents")

print(f"\n{'='*60}")
print("BUSINESS INTERPRETATION:")
print(f"{'='*60}")
print(f"Point Estimate: Expect ~{int(next_day_forecast)} documents tomorrow")
print(f"\nPlanning Scenarios:")
print(f"  - Conservative (95th percentile): Plan for {ci_upper} documents")
print(f"  - Most Likely: Plan for {int(next_day_forecast)} documents")
print(f"  - Optimistic (5th percentile): Could be as low as {ci_lower} documents")

uncertainty_pct = (ci_width / next_day_forecast) * 100
print(f"\nUncertainty: ±{uncertainty_pct:.1f}% around the point estimate")

if uncertainty_pct < 20:
    print("  → Low uncertainty. High confidence in forecast.")
elif uncertainty_pct < 40:
    print("  → Moderate uncertainty. Plan with some buffer.")
else:
    print("  → High uncertainty. Consider gathering more data or features.")

print(f"\n{'='*60}")
print("NEXT STEPS:")
print(f"{'='*60}")
print("1. Use point estimate for baseline capacity planning")
print("2. Use upper bound for worst-case resource allocation")
print("3. Monitor actual ingestion tomorrow and compare to forecast")
print("4. Retrain model weekly with new data to maintain accuracy")

In [0]:
# ============================================================
# EXTENDED FORECAST: Predict Multiple Days Ahead
# ============================================================
# Generate forecasts for the next 7 days with confidence intervals

print("Generating 7-day forecast...\n")

# Forecast 7 days ahead
forecast_steps = 7
forecast_result = final_model_fit.get_forecast(steps=forecast_steps)
forecast_values = forecast_result.predicted_mean
forecast_ci = forecast_result.conf_int(alpha=0.05)

# Create forecast dates
forecast_dates = [df_ts.index[-1] + timedelta(days=i+1) for i in range(forecast_steps)]

# Display results
print("="*70)
print("7-DAY FORECAST")
print("="*70)
for i, date in enumerate(forecast_dates):
    pred = int(forecast_values.iloc[i])
    lower = int(forecast_ci.iloc[i, 0])
    upper = int(forecast_ci.iloc[i, 1])
    print(f"{date.strftime('%Y-%m-%d (%A)')}: {pred:3d} docs (95% CI: [{lower:3d}, {upper:3d}])")

# Visualize extended forecast
plt.figure(figsize=(14, 6))

# Plot historical data (last 60 days for context)
historical_window = df_ts[-60:]
plt.plot(historical_window.index, historical_window['number_documents'], 
         label='Historical Data', linewidth=2, color='#2E86AB')

# Plot forecast
plt.plot(forecast_dates, forecast_values, 
         label='Forecast', marker='o', linewidth=2, markersize=7, 
         color='#A23B72', linestyle='--')

# Plot confidence interval
plt.fill_between(forecast_dates, 
                 forecast_ci.iloc[:, 0], 
                 forecast_ci.iloc[:, 1], 
                 alpha=0.2, color='#A23B72', label='95% Confidence Interval')

plt.title('7-Day Document Ingestion Forecast', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Number of Documents', fontsize=12)
plt.legend(fontsize=11, loc='upper left')
plt.grid(True, alpha=0.3, linestyle=':')
plt.tight_layout()
plt.show()

print("\n✓ Extended forecast complete!")
print("\nNote: Uncertainty increases for forecasts further into the future.")
print("Confidence intervals widen as we predict days 2, 3, 4... ahead.")